# Analyzing Eye Tracking (Gaze) Data

**CS474: Human Computer Interaction — Modalities: Eye Tracking**

Eye trackers report a stream of `(x, y)` gaze positions over time.  Before we can use gaze as an input modality, we need to turn that raw stream into meaningful events like **fixations** (moments when the eye pauses on something interesting) and **saccades** (rapid jumps between fixations).

In this notebook you will:

1. Generate a *synthetic* gaze recording, so you can experiment without any eye tracking hardware
2. Detect fixations using a simple **dispersion threshold** algorithm (I-DT)
3. Visualize gaze as a scanpath and as a heatmap — the two most common visualizations in eye tracking studies

No webcam or special hardware is required — everything runs on synthetic data using `numpy` and `matplotlib`, which are pre-installed on Google Colab.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(474)  # seeded so everyone sees the same data

## Part 1: Simulate a Gaze Recording

Imagine a user looking at a 1280x720 web page with three "areas of interest" (AOIs): a logo, a navigation menu, and a call-to-action button.  A real eye tracker samples gaze at 60-1000 Hz; we'll simulate 60 Hz.

During a **fixation**, gaze samples cluster tightly around one point (with a little jitter from the eye's microtremor and tracker noise).  During a **saccade**, the eye sweeps quickly to the next target.

In [ ]:
SCREEN_W, SCREEN_H = 1280, 720
HZ = 60  # samples per second

# (x, y, seconds the user dwells there)
aois = [
    (150, 100, 0.8),   # logo, top-left
    (640, 100, 0.5),   # navigation menu, top-center
    (640, 360, 1.2),   # main content
    (1050, 600, 0.9),  # call-to-action button, bottom-right
    (640, 360, 0.6),   # back to the content
]

samples = []
for (x, y, dwell) in aois:
    n = int(dwell * HZ)
    # fixation: tight Gaussian jitter around the AOI center
    fx = rng.normal(x, 8, n)
    fy = rng.normal(y, 8, n)
    samples.append(np.column_stack([fx, fy]))
    # saccade: a few fast samples on the way to the next target (we just leave a gap)

gaze = np.vstack(samples)
t = np.arange(len(gaze)) / HZ
print(f"{len(gaze)} gaze samples over {t[-1]:.2f} seconds")

## Part 2: Detect Fixations with a Dispersion Threshold (I-DT)

The classic **I-DT algorithm** (Salvucci & Goldberg, 2000) slides a window over the samples and declares a fixation whenever the *dispersion* — `(max(x) - min(x)) + (max(y) - min(y))` — stays under a threshold for at least a minimum duration (commonly ~100 ms).

In [ ]:
def detect_fixations(points, hz, dispersion_px=50, min_duration_s=0.10):
    """Return a list of (start_idx, end_idx, centroid_x, centroid_y, duration_s)."""
    min_len = int(min_duration_s * hz)
    fixations = []
    i = 0
    while i < len(points) - min_len:
        j = i + min_len
        window = points[i:j]
        disp = (np.ptp(window[:, 0])) + (np.ptp(window[:, 1]))
        if disp <= dispersion_px:
            # grow the window while dispersion stays small
            while j < len(points):
                window = points[i:j + 1]
                if (np.ptp(window[:, 0])) + (np.ptp(window[:, 1])) > dispersion_px:
                    break
                j += 1
            cx, cy = points[i:j].mean(axis=0)
            fixations.append((i, j, cx, cy, (j - i) / hz))
            i = j
        else:
            i += 1
    return fixations

fixations = detect_fixations(gaze, HZ)
for k, (i, j, cx, cy, dur) in enumerate(fixations, 1):
    print(f"Fixation {k}: ({cx:6.1f}, {cy:6.1f})  duration {dur*1000:5.0f} ms")

## Part 3: Visualize the Scanpath

A **scanpath** plot shows fixations as circles (sized by duration) connected in order — it tells the story of *where the user looked, in what order, and for how long*.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(gaze[:, 0], gaze[:, 1], '.', ms=2, alpha=0.3, label='raw gaze samples')

cx = [f[2] for f in fixations]
cy = [f[3] for f in fixations]
durs = [f[4] for f in fixations]
ax.plot(cx, cy, '-', color='gray', lw=1)
ax.scatter(cx, cy, s=[d * 800 for d in durs], alpha=0.6, zorder=3, label='fixations')
for k, (x, y) in enumerate(zip(cx, cy), 1):
    ax.annotate(str(k), (x, y), ha='center', va='center', fontweight='bold')

ax.set_xlim(0, SCREEN_W); ax.set_ylim(SCREEN_H, 0)  # invert y: screen coords
ax.set_title('Scanpath: fixation order and duration')
ax.legend(loc='lower left')
plt.show()

## Part 4: Visualize a Gaze Heatmap

Heatmaps aggregate attention over time — the brighter a region, the more gaze samples landed there.  Designers use these to check whether users actually notice the elements the design intends to emphasize.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
h = ax.hist2d(gaze[:, 0], gaze[:, 1], bins=[64, 36],
              range=[[0, SCREEN_W], [0, SCREEN_H]], cmap='hot')
ax.set_ylim(SCREEN_H, 0)
ax.set_title('Gaze heatmap (attention density)')
fig.colorbar(h[3], ax=ax, label='samples per bin')
plt.show()

## Your Turn

1. **Change the noise.**  Increase the jitter standard deviation from 8 px to 30 px in Part 1 (as if the tracker were poorly calibrated).  What happens to the number of detected fixations?  How would you adjust `dispersion_px` to compensate — and what could that trade off?
2. **Add a distraction.**  Add a new AOI representing a blinking ad in the corner that the user glances at for 150 ms.  Does I-DT catch it with the default `min_duration_s`?  Should it?
3. **Design connection.**  Suppose the heatmap showed users never look at the call-to-action button.  List two design changes (from our discussions of signifiers and visual hierarchy) that could redirect attention — and one *dark pattern* you should avoid.

## Reflection

Real eye trackers (including webcam-based ones like [WebGazer.js](https://webgazer.cs.brown.edu/)) add calibration error, blinks, and dropped samples on top of what we simulated.  As you work on the eye tracking programming assignment, think about how thresholds like these must adapt to each user and lighting condition.